# Open in Colab — 3D Craniofacial Pipeline

Runs the existing prototype on all `.nii` / `.nii.gz` scans in `My Drive/new_data/`:

**NIfTI → TotalSegmentator (craniofacial_structures + teeth) → 3D masks → axial/coronal/sagittal projections → pairwise Dice/IoU comparison.**

Same-person scan pairs are excluded. Different-person scans are compared pairwise.

## 1. Runtime check

Select **Runtime → Change runtime type → GPU** first. The assigned GPU is recorded.

In [ ]:
import os, sys, platform, subprocess
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
except Exception as e:
    print("PyTorch not ready yet:", e)
subprocess.run(["nvidia-smi"], check=False)

## 2. Clone and install

The runner uses the repository's pinned TotalSegmentator version and the existing `fast=False` settings.

In [ ]:
!rm -rf /content/3D-Craniofacial-Pipeline
!git clone -q -b feat/colab-runner https://github.com/trungnb/3D-Craniofacial-Pipeline.git /content/3D-Craniofacial-Pipeline
!python -m pip install -q --upgrade pip
!python -m pip install -q -r /content/3D-Craniofacial-Pipeline/requirements.txt
import totalsegmentator, torch
print("TotalSegmentator:", getattr(totalsegmentator, "__version__", "installed"))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

## 3. Google Drive + input data

Put the CT/CBCT files in `My Drive/new_data/`.

If multiple scans are from the **same person**, put them in one subfolder, e.g. `new_data/person01/scan1.nii.gz` and `new_data/person01/scan2.nii.gz`. Same-person pairs are skipped.

If all files are directly inside `new_data/`, each file is treated as a different person.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, time, json, itertools
from datetime import datetime, timezone
from collections import Counter

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_DIR = DRIVE_ROOT / "new_data"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_ROOT = DRIVE_ROOT / "new_data_runs" / RUN_ID
LOCAL_ROOT = Path("/content/cbct_pipeline")
LOCAL_INPUT = LOCAL_ROOT / "input"
LOCAL_OUTPUT = LOCAL_ROOT / "output"

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Create {DATA_DIR} and add .nii/.nii.gz files first.")

nii_files = sorted(p for p in DATA_DIR.rglob("*") if p.is_file() and (p.name.endswith(".nii") or p.name.endswith(".nii.gz")))
if not nii_files:
    raise FileNotFoundError(f"No NIfTI files found under {DATA_DIR}.")

cases = []
for src in nii_files:
    rel = src.relative_to(DATA_DIR)
    person_id = src.stem.replace(".nii", "") if len(rel.parts) == 1 else rel.parts[0]
    scan_id = src.name.replace(".nii.gz", "").replace(".nii", "")
    cases.append({"source": src, "person_id": person_id, "scan_id": scan_id})

print("Found", len(cases), "scan(s)")
print("Person groups:")
for person, n in Counter(c["person_id"] for c in cases).items():
    print(f"  {person}: {n} scan(s)")

## 4. Run segmentation + projections

Each scan is processed **one at a time** in Colab scratch space. Drive keeps the outputs.

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from totalsegmentator.python_api import totalsegmentator

RUN_ROOT.mkdir(parents=True, exist_ok=False)
summary = []

def make_projections(mask_path, out_dir):
    img = nib.load(str(mask_path))
    mask = (img.get_fdata() > 0).astype(np.uint8)
    out_dir.mkdir(parents=True, exist_ok=True)
    plt.imsave(out_dir/"axial_projection.png", np.max(mask, axis=2), cmap="gray")
    plt.imsave(out_dir/"coronal_projection.png", np.max(mask, axis=1), cmap="gray")
    plt.imsave(out_dir/"sagital_projection.png", np.max(mask, axis=0), cmap="gray")

for case in cases:
    src = case["source"]
    name = case["scan_id"]
    person_id = case["person_id"]
    print(f"\n=== {name} | person={person_id} ===")

    shutil.rmtree(LOCAL_ROOT, ignore_errors=True)
    LOCAL_INPUT.mkdir(parents=True, exist_ok=True)
    LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)

    local_src = LOCAL_INPUT / src.name
    shutil.copy2(src, local_src)
    seg = LOCAL_OUTPUT / "Volumes" / name
    proj = LOCAL_OUTPUT / "Projections" / name
    seg.mkdir(parents=True, exist_ok=True)

    t0 = time.time()
    totalsegmentator(input=str(local_src), output=str(seg), task="craniofacial_structures", output_type="niftis", fast=False)
    totalsegmentator(input=str(local_src), output=str(seg), task="teeth", output_type="niftis", fast=False)
    seg_time = time.time() - t0

    p0 = time.time()
    masks = sorted(seg.glob("*.nii*"))
    for mask in masks:
        make_projections(mask, proj / mask.name.replace(".nii.gz", "").replace(".nii", ""))
    proj_time = time.time() - p0

    case_root = RUN_ROOT / "cases" / name
    (case_root / "input").mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, case_root / "input" / src.name)
    shutil.copytree(seg, case_root / "output" / "Volumes", dirs_exist_ok=True)
    shutil.copytree(proj, case_root / "output" / "Projections", dirs_exist_ok=True)

    summary.append({"scan":name, "person_id":person_id, "segmentation_seconds":seg_time, "projection_seconds":proj_time, "mask_files":len(masks)})
    print(f"Completed: segmentation={seg_time:.1f}s, projections={proj_time:.1f}s, masks={len(masks)}")

    shutil.rmtree(LOCAL_ROOT, ignore_errors=True)

## 5. Pairwise comparison — existing prototype logic

- **Same person:** skipped.
- **Different people:** every scan is compared with every other different-person scan. With 50 scans from 50 people this is **1 vs 49 for each scan = 1,225 unique pairs**.
- **3D:** second mask is resampled to the first mask's reference grid, then Dice/IoU are calculated.
- **2D:** second projection is resized to the first projection's dimensions when needed, then Dice/IoU are calculated.

This is the exploratory between-case comparison from the existing prototype; it is not anatomical registration or identification-accuracy validation.

In [ ]:
import cv2
import SimpleITK as sitk
import pandas as pd

def score_metrics(mask1, mask2):
    vol1, vol2 = np.sum(mask1), np.sum(mask2)
    if vol1 == 0 and vol2 == 0:
        dice = 1.0
    elif vol1 == 0 or vol2 == 0:
        dice = 0.0
    else:
        dice = 2 * np.sum(mask1 & mask2) / (vol1 + vol2)
    union = np.sum(mask1 | mask2)
    iou = 1.0 if union == 0 else np.sum(mask1 & mask2) / union
    return float(dice), float(iou)

def compare_nii(path1, path2):
    img1 = sitk.ReadImage(str(path1))
    img2 = sitk.ReadImage(str(path2))
    img2 = sitk.Resample(img2, img1, sitk.Transform(), sitk.sitkNearestNeighbor, 0, img2.GetPixelID())
    return score_metrics(sitk.GetArrayFromImage(img1) > 0, sitk.GetArrayFromImage(img2) > 0)

def compare_png(path1, path2):
    img1 = cv2.imread(str(path1), cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(str(path2), cv2.IMREAD_GRAYSCALE)
    if img1 is None or img2 is None:
        raise ValueError("Cannot read projection PNG")
    mask1, mask2 = (img1 > 0).astype(np.uint8), (img2 > 0).astype(np.uint8)
    if mask1.shape != mask2.shape:
        mask2 = cv2.resize(mask2, (mask1.shape[1], mask1.shape[0]), interpolation=cv2.INTER_NEAREST)
    return score_metrics(mask1, mask2)

case_dirs = {c["scan_id"]: RUN_ROOT/"cases"/c["scan_id"]/"output" for c in cases}
case_people = {c["scan_id"]: c["person_id"] for c in cases}
scan_ids = sorted(case_dirs)
pair_rows_3d, pair_rows_2d = [], []

for scan1, scan2 in itertools.combinations(scan_ids, 2):
    if case_people[scan1] == case_people[scan2]:
        continue

    v1, v2 = case_dirs[scan1]/"Volumes"/scan1, case_dirs[scan2]/"Volumes"/scan2
    common_masks = sorted(set(p.name for p in v1.glob("*.nii*")) & set(p.name for p in v2.glob("*.nii*")))
    for fname in common_masks:
        t0 = time.time()
        try:
            dice, iou = compare_nii(v1/fname, v2/fname)
            pair_rows_3d.append({"scan1":scan1,"scan2":scan2,"person1":case_people[scan1],"person2":case_people[scan2],"volume":fname,"dice_score":dice,"iou_score":iou,"time":time.time()-t0})
        except Exception as e:
            print("3D comparison skipped:", scan1, scan2, fname, e)

    p1, p2 = case_dirs[scan1]/"Projections"/scan1, case_dirs[scan2]/"Projections"/scan2
    common_volumes = sorted(set(p.name for p in p1.iterdir() if p.is_dir()) & set(p.name for p in p2.iterdir() if p.is_dir()))
    for volume_name in common_volumes:
        common_pngs = sorted(set(p.name for p in (p1/volume_name).glob("*.png")) & set(p.name for p in (p2/volume_name).glob("*.png")))
        for fname in common_pngs:
            t0 = time.time()
            try:
                dice, iou = compare_png(p1/volume_name/fname, p2/volume_name/fname)
                pair_rows_2d.append({"scan1":scan1,"scan2":scan2,"person1":case_people[scan1],"person2":case_people[scan2],"volume":volume_name,"projection":fname,"dice_score":dice,"iou_score":iou,"time":time.time()-t0})
            except Exception as e:
                print("2D comparison skipped:", scan1, scan2, volume_name, fname, e)

volume_between_case_result = pd.DataFrame(pair_rows_3d)
projection_between_case_result = pd.DataFrame(pair_rows_2d)
print("3D comparison rows:", len(volume_between_case_result))
print("2D comparison rows:", len(projection_between_case_result))

## 6. Save comparison results

In [ ]:
summary_df = pd.DataFrame(summary)
summary_df.to_csv(RUN_ROOT/"run_summary.csv", index=False)
volume_between_case_result.to_csv(RUN_ROOT/"volume_between_case_results.csv", index=False)
projection_between_case_result.to_csv(RUN_ROOT/"projection_between_case_results.csv", index=False)

combined_rows = []
if not volume_between_case_result.empty:
    combined_rows.append({"comparison_type":"Volumes (different-person pairs)","avg_dice_score":volume_between_case_result.dice_score.mean(),"avg_iou_score":volume_between_case_result.iou_score.mean(),"total_comparison_time_seconds":volume_between_case_result.time.sum()})
if not projection_between_case_result.empty:
    combined_rows.append({"comparison_type":"Projections (different-person pairs)","avg_dice_score":projection_between_case_result.dice_score.mean(),"avg_iou_score":projection_between_case_result.iou_score.mean(),"total_comparison_time_seconds":projection_between_case_result.time.sum()})
combined_results = pd.DataFrame(combined_rows)
combined_results.to_csv(RUN_ROOT/"between_case_summary.csv", index=False)

torch = __import__("torch")
manifest = {
    "run_id":RUN_ID,
    "n_scans":len(cases),
    "n_persons":len(set(c["person_id"] for c in cases)),
    "n_unique_different_person_pairs":sum(1 for a,b in itertools.combinations(scan_ids,2) if case_people[a] != case_people[b]),
    "python":sys.version,
    "pytorch":torch.__version__,
    "cuda_available":bool(torch.cuda.is_available()),
    "gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "totalsegmentator":getattr(totalsegmentator,"__version__","2.18.0"),
    "tasks":["craniofacial_structures","teeth"],
    "fast":False,
    "comparison":"different-person pairwise; same-person pairs excluded; 3D resample-to-reference and 2D nearest-neighbor resize",
    "timestamp_utc":datetime.now(timezone.utc).isoformat()
}
(RUN_ROOT/"run_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
display(summary_df)
display(combined_results)

## 7. Quick visual check

In [ ]:
from IPython.display import Image, display
first = scan_ids[0]
pngs = sorted((RUN_ROOT/"cases"/first/"output"/"Projections"/first).glob("*/*.png"))
for p in pngs[:6]:
    print(p.relative_to(RUN_ROOT))
    display(Image(filename=str(p), width=360))

### Output

Saved under `My Drive/new_data_runs/<run_id>/`.

- `run_summary.csv` — per-scan segmentation/projection runtime
- `volume_between_case_results.csv` — 3D Dice/IoU for every different-person pair and common mask
- `projection_between_case_results.csv` — 2D Dice/IoU for every different-person pair and projection
- `between_case_summary.csv` — overall mean Dice/IoU and comparison time
- `run_manifest.json` — runtime and comparison configuration

The between-case scores remain the same exploratory prototype measure described in the repository README.